# 03 · Parte 2: RAG, herramientas y agente

Este notebook no necesita llave de API salvo la última celda (opcional).

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 140)
import json
from aire import config as C
from aire.rag import Retriever, load_corpus

## 1. Corpus y fragmentación

In [2]:
chunks = load_corpus()
print(len(chunks), 'fragmentos')
pd.Series([c.doc_id for c in chunks]).value_counts()

56 fragmentos


RES2254    14
SALUD       9
POECA       8
METEO       7
METODO      6
OMS2021     6
SIATA       6
Name: count, dtype: int64

## 2. Recuperación: híbrido (FAISS + TF-IDF con RRF) vs. solo embeddings vs. solo TF-IDF

In [3]:
print((C.REPORTS_DIR / 'evaluacion_rag.md').read_text(encoding='utf-8'))

| Recuperador | hit@1 | hit@4 | MRR |
|---|---|---|---|
| hibrido | 0.72 | 1.00 | 0.84 |
| embeddings+faiss | 0.78 | 0.94 | 0.85 |
| tfidf | 0.72 | 0.94 | 0.80 |



In [4]:
ret = Retriever()
for r in ret.search('¿Qué deben hacer los grupos sensibles con ICA naranja?', k=3):
    print(r['score'], r['doc_id'], '|', r['seccion'])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5909.48it/s]

1.9513 RES2254 | Índice de Calidad del Aire - ICA (Artículos 18 y 19, Tabla 5)
1.9211 SALUD | ICA Naranja - Dañina a la salud de grupos sensibles (PM2.5 de 38 a 55 µg/m3)
1.833 SALUD | ICA Rojo - Dañina a la salud (PM2.5 de 56 a 150 µg/m3)


## 3. Herramientas (tool calling) y manejo de errores
Los errores se devuelven como observación `{ok: false, error: ...}` en lugar de romper el loop.

In [5]:
from aire.tools import call_tool, TOOL_NAMES
print(TOOL_NAMES)
for name, args in [('get_pm25', {'estacion': 'Belén'}), ('predict_tomorrow', {'estacion': 'MED-BEME'}),
                   ('get_episode_state', {}), ('get_pm25', {'estacion': 'Rionegro'}), ('get_pm25', {'estacion': 'Bello', 'fecha': '2025-01-15'})]:
    out = call_tool(name, args, default_date='2024-03-15')
    print(f'\n{name}{args} ok={out["ok"]}')
    print(json.dumps(out, ensure_ascii=False)[:400])

['get_pm25', 'predict_tomorrow', 'get_episode_state', 'search_docs']

get_pm25{'estacion': 'Belén'} ok=True
{"ok": true, "result": {"fecha_consultada": "2024-03-15", "resumen": null, "estaciones": [{"codigo": 83, "nombre_corto": "MED-BEME", "nombre": "Medellin, Belén - I.E Pedro Justo Berrio", "municipio": "Medellín", "latitud": 6.2372341, "longitud": -75.610466, "fecha_dato": "2024-03-15", "pm25_promedio_24h_ugm3": 27.4, "ica": 80, "categoria_ica": "Aceptable", "pm25_max_horario_ugm3": 43.6, "horas_val



predict_tomorrow{'estacion': 'MED-BEME'} ok=True
{"ok": true, "result": {"estacion": {"codigo": 83, "nombre_corto": "MED-BEME", "nombre": "Medellin, Belén - I.E Pedro Justo Berrio", "municipio": "Medellín", "latitud": 6.2372341, "longitud": -75.610466}, "fecha_base": "2024-03-15", "fecha_pronostico": "2024-03-16", "pm25_hoy_ugm3": 27.4, "categoria_hoy": "Aceptable", "probabilidades_manana": {"Buena": 0.008, "Aceptable": 0.912, "Dañina (grupos se

get_episode_state{} ok=True
{"ok": true, "result": {"fecha": "2024-03-15", "estaciones_por_estado_hmm": {"Episodio crítico": 15, "Alerta": 2}, "estaciones_en_rango_prevencion_o_mas (>=38 µg/m3)": 1, "n_estaciones": 17, "detalle": [{"estacion": "BAR-TORR", "municipio": "Barbosa", "pm25": 28.9, "categoria_ica": "Aceptable", "estado_mas_probable": "Episodio crítico"}, {"estacion": "BEL-FEVE", "municipio": "Bello", "pm25": 28.3,

get_pm25{'estacion': 'Rionegro'} ok=False
{"ok": false, "error": "No encontré la estación o municipio 'Rionegro'. Esta

## 4. Loop agéntico
`src/aire/agent.py`: el LLM recibe el prompt de sistema con el contexto (fecha y estación del dashboard) y los esquemas de las 4 herramientas; en cada iteración (máximo `T_MAX = 6`) decide llamar herramientas o responder. Si se alcanza `T_MAX` se fuerza una respuesta final sin herramientas. Los errores de API (llave, límite de uso, red) se convierten en mensajes claros.

La evaluación con 14 preguntas fijas está en `scripts/evaluate_agent.py` y sus resultados en `evaluacion/`.

In [6]:
import os
from aire.agent import llm_config
cfg = llm_config()
print('Proveedor:', cfg['provider'], '| modelo:', cfg['model'], '| llave:', bool(cfg['api_key']))
if cfg['api_key'] and cfg['provider'] != 'ollama':
    from aire.agent import AirQualityAgent
    res = AirQualityAgent().run('¿Cómo está el aire hoy en Belén y qué significa?', fecha='2024-03-15', estacion='MED-BEME')
    print(res.tools_called); print(res.answer)
else:
    print('Sin llave de API: ver README para configurar .env')

Proveedor: groq | modelo: openai/gpt-oss-120b | llave: False
Sin llave de API: ver README para configurar .env
